# residual-stream-dynamics — Colab runnerRuns the corpus workflows on a free-tier GPU. Execute the setup cells(1–5) in order once per session, then run whichever analysis cells you need.**Free-tier notes**- The GPU is a **T4 (16GB)**. `gpt2-small` through `pythia-2.8b` all fit;  `pythia-6.9b` does **not** and is not usable here.- The runtime is recycled after ~90 min idle / 12 h max, and `/content` is  wiped with it. Cell 4 mounts Drive so `data/` and `figures/` survive.- Set the runtime to GPU first: **Runtime → Change runtime type → T4 GPU**.

## 1. Verify the GPU is attached

In [ ]:
!nvidia-smiimport torchprint()print(f"torch {torch.__version__}")print(f"cuda available: {torch.cuda.is_available()}")if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print(f"device: {p.name}  ({p.total_memory / 1024**3:.1f} GB)")else:    print("NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun.")

## 2. Clone the repository`data/` and `figures/` are gitignored, so this clone is small (a few MB).Results are written to Drive in cell 4.

In [ ]:
import osfrom pathlib import PathREPO_URL  = "https://github.com/storyofthewolf/residual-stream-dynamics.git"REPO_NAME = "residual-stream-dynamics"REPO_DIR  = Path("/content") / REPO_NAMEif REPO_DIR.exists():    print(f"{REPO_DIR} already exists — pulling latest")    !cd {REPO_DIR} && git pull --ff-onlyelse:    !git clone {REPO_URL} {REPO_DIR}os.chdir(REPO_DIR)print(f"\ncwd: {Path.cwd()}")

## 3. Install dependenciesColab ships its own torch build — we deliberately do **not** reinstall it, sincepip would pull a CPU-only or CUDA-mismatched wheel and break GPU support.Only the packages Colab lacks are installed.

In [ ]:
# Install everything EXCEPT torch (Colab's preinstalled build is CUDA-matched).!pip install -q transformer_lens sae_lensimport torchprint(f"\ntorch {torch.__version__} — cuda available: {torch.cuda.is_available()}")assert torch.cuda.is_available(), "GPU lost after install — Runtime > Restart, then rerun."

## 4. Mount Drive and redirect outputs`data/` and `figures/` become symlinks into Drive, so every `.npz` and `.png`the workflows write persists across runtime restarts. Nothing in the repo codechanges — the workflows still resolve paths from `_PROJECT_ROOT`, which nowpoints through the symlink.

In [ ]:
from google.colab import drivefrom pathlib import Pathimport shutil, osdrive.mount("/content/drive")# Everything lives under one Drive folder; change if you prefer another location.DRIVE_ROOT = Path("/content/drive/MyDrive/residual-stream-dynamics")(DRIVE_ROOT / "data").mkdir(parents=True, exist_ok=True)(DRIVE_ROOT / "figures").mkdir(parents=True, exist_ok=True)REPO_DIR = Path("/content/residual-stream-dynamics")for name in ("data", "figures"):    local = REPO_DIR / name    target = DRIVE_ROOT / name    if local.is_symlink():        local.unlink()    elif local.exists():        # A fresh clone has an empty dir (just .gitkeep) — safe to replace.        shutil.rmtree(local)    local.symlink_to(target)    print(f"{local}  ->  {target}")print("\nOutputs will persist in Drive across runtime restarts.")

## 5. Smoke testConfirms the model loads on CUDA and the pipeline runs end to end before youcommit to a long job.

In [ ]:
!python workflows/single_prompt.py --model gpt2-small --hooks resid_post \    --logit-lens --no-plots --alpha 1.0

---## Analysis workflowsEach cell is independent. `--device` now defaults to auto-detect, so CUDA ispicked up automatically — no flag needed.Add `--run-tag <name>` to keep successive runs from overwriting each other.

### Entropy analysis

In [ ]:
!python workflows/entropy_analysis.py --model gpt2-small \    --corpus corpus/base_vs_contrast_n216.json --save-data

### Ablation analysis (the primary experiment)

In [ ]:
!python workflows/ablation_analysis.py --model gpt2-small \    --corpus corpus/base_vs_contrast_n216.json \    --ev-thresholds 0.50 0.75 0.90 0.95 0.99 \    --save-data

### c_k spectrum analysis`--last-token-only` stores `[n_layers, 1, d_model]` instead of the full tokenaxis, shrinking the `.npz` by roughly the sequence length. Eight of the ninec_k figures use only the final token, so this discards nothing they need; theall-tokens heatmap is skipped automatically.

In [ ]:
!python workflows/ck_analysis.py --model gpt2-small \    --corpus corpus/base_vs_contrast_n216.json \    --last-token-only --save-data

### W_U subspace analysis

In [ ]:
!python workflows/wu_subspace_analysis.py --model gpt2-small \    --corpus corpus/base_vs_contrast_n216.json --save-data

### Mechanics analysis

In [ ]:
!python workflows/mechanics_analysis.py --model gpt2-small \    --corpus corpus/base_vs_contrast_n216.json --save-data

---## Sweeping models`pythia-2.8b` and `gpt2-xl` load in float16 automatically on CUDA so they fitthe T4. `pythia-6.9b` is deliberately excluded — it does not fit in 16GB.Run this unattended; results accumulate in Drive. If the runtime dies partway,completed models are already saved and you can restart from the survivors.

In [ ]:
MODELS = ["gpt2-small", "gpt2-medium", "gpt2-large", "pythia-160m", "pythia-1b"]for m in MODELS:    print(f"\n{'='*70}\n  {m}\n{'='*70}")    !python workflows/ablation_analysis.py --model {m} \        --corpus corpus/base_vs_contrast_n216.json \        --ev-thresholds 0.50 0.75 0.90 0.95 0.99 --save-data --no-plots

### Larger models (float16, one at a time)Run these individually and restart the runtime between them to release VRAM.

In [ ]:
!python workflows/ablation_analysis.py --model pythia-2.8b \    --corpus corpus/base_vs_contrast_n216.json \    --ev-thresholds 0.50 0.90 0.99 --save-data --no-plots

---## Free VRAM between modelsColab does not release GPU memory until the process exits. The `!python`invocations above each run in their own process, so VRAM is freed automaticallywhen they finish. Use this only if you loaded a model inside the notebook itself.

In [ ]:
import gc, torchgc.collect()torch.cuda.empty_cache()print(f"VRAM allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")print(f"VRAM reserved:  {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

---## Check what has been saved to Drive

In [ ]:
!du -sh /content/drive/MyDrive/residual-stream-dynamics/data/* 2>/dev/null!echo "---"!ls -lh /content/drive/MyDrive/residual-stream-dynamics/data/ablation/ 2>/dev/null | tail -20